In [2]:
import pandas as pd
import json

# ================= LOAD =================
df = pd.read_csv(
    "/kaggle/input/datasets/pulkitchatwal/fincausal/train_en_2000 (2).csv",
    engine="python",        # more flexible parser
    sep=";",                # default but explicit
    quotechar='"',          # handle quotes correctly
    on_bad_lines="skip"     # skip corrupted rows (important)
)

# ================= CLEAN =================
df["context"] = df["context"].astype(str).str.strip()
df["question"] = df["question"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()

# Remove empty rows
df = df[(df["context"] != "") & (df["question"] != "") & (df["answer"] != "")]

# ================= SYSTEM PROMPT =================
SYSTEM_PROMPT = """
You are a financial causality extraction assistant for the FinCausal task.

Goal:
Extract the exact span from the context that answers the question.

Rules:
- The question asks about a CAUSAL relationship (cause or effect).
- Return ONLY the exact phrase from the context.
- Do NOT paraphrase.
- Do NOT explain.
- Do NOT add new words.
- Do NOT hallucinate.
- The answer must be verbatim from the context.
- Prefer the minimal span that fully answers the question.
"""

# ================= BUILD EXAMPLE =================
def build_example(row):
    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT.strip()
            },
            {
                "role": "user",
                "content": f"Context:\n{row['context']}\n\nQuestion:\n{row['question']}"
            },
            {
                "role": "assistant",
                "content": row["answer"]
            }
        ]
    }

# ================= WRITE JSONL =================
with open("train_qwen_fincausal.jsonl", "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        example = build_example(row)
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("✅ FinCausal-optimized Qwen training file created → train_qwen_fincausal.jsonl")

✅ FinCausal-optimized Qwen training file created → train_qwen_fincausal.jsonl
